# Download Historical Data

Fetches OHLCV trendbar data from the cTrader Open API and saves it to `data/`.

`Learn.data.fetch_ohlcv()` is the single-symbol helper. For multi-symbol downloads in one run, use `Learn.data.fetch_ohlcv_bulk()`.

> **Note:** The Twisted reactor can only run once per kernel session.  
> If you need to re-download, **restart the kernel** before re-running.

In [ ]:
# -- Configuration -----------------------------------------------------------
symbol_name     = "US500"   # Symbol name (must match broker exactly)
num_chunks      = 52 * 10   # Number of time chunks to fetch
weeks_per_chunk = 1         # Width of each chunk in weeks
period_str      = "M1"      # Bar period: M1 M2 M3 M4 M5 M10 M15 M30 H1 H4 H12 D1 W1 MN1

In [ ]:
import pathlib
import pandas as pd

from Learn.data import fetch_ohlcv, fetch_ohlcv_bulk

df = fetch_ohlcv(
    symbol_name     = symbol_name,
    num_chunks      = num_chunks,
    weeks_per_chunk = weeks_per_chunk,
    period_str      = period_str,
    save_csv        = False,
)

## Validation

In [ ]:
# -- Basic info --------------------------------------------------------------
print(f"Shape      : {df.shape}")
print(f"Date range : {df['Time'].min()}  ->  {df['Time'].max()}")
print(f"Dtypes     :\n{df.dtypes}")
df.head(3)

In [ ]:
# -- Null check --------------------------------------------------------------
null_counts = df.isnull().sum()
print("Null counts per column:")
print(null_counts)
assert null_counts.sum() == 0, "Unexpected nulls detected -- check fetch output."

In [ ]:
# -- Duplicate timestamps ----------------------------------------------------
n_dupes = df.duplicated(subset='Time').sum()
print(f"Duplicate timestamps : {n_dupes}")
assert n_dupes == 0, "Duplicate timestamps found -- check fetch output."

In [ ]:
# -- OHLC sanity -------------------------------------------------------------
ohlc_violations = (
    (df['High'] < df['Low']).sum()   +
    (df['High'] < df['Open']).sum()  +
    (df['High'] < df['Close']).sum() +
    (df['Low']  > df['Open']).sum()  +
    (df['Low']  > df['Close']).sum()
)
print(f"OHLC violations : {ohlc_violations}")
assert ohlc_violations == 0, "OHLC relationship violations detected."

In [ ]:
# -- Gap analysis ------------------------------------------------------------
PERIOD_MINUTES = {
    'M1': 1, 'M2': 2, 'M3': 3, 'M4': 4, 'M5': 5, 'M10': 10,
    'M15': 15, 'M30': 30, 'H1': 60, 'H4': 240, 'H12': 720,
    'D1': 1440, 'W1': 10080,
}
if period_str in PERIOD_MINUTES:
    expected_delta = pd.Timedelta(minutes=PERIOD_MINUTES[period_str])
    times = pd.to_datetime(df['Time'])
    deltas = times.diff().dropna()
    gaps = deltas[deltas > expected_delta * 1.5]
    print(f"Gaps > 1.5x bar period ({expected_delta}): {len(gaps)}")
    if len(gaps) > 0:
        print(gaps.sort_values(ascending=False).head(10))
else:
    print(f"Gap analysis skipped for period '{period_str}'.")

In [ ]:
# -- Price distribution ------------------------------------------------------
df[['Open', 'High', 'Low', 'Close', 'Volume']].describe().round(6)

## Save

In [ ]:
total_weeks = num_chunks * weeks_per_chunk
output_path = pathlib.Path("../data") / f"{symbol_name}_{period_str}_{total_weeks}weeks.csv"
output_path = output_path.resolve()

df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} rows -> {output_path}")